# 5.6 學習率怎麼隨訓練變化？


一趟40步訓練，可以先小步起步，再放大，最後細調。高峰0.001、前五步暖身時，前五次依序是0.0002、0.0004、0.0006、0.0008、0.001。這份隨步數給步幅的時間表叫排程。

若不想起步幾次就大幅改動參數，可以先縮小這幾次的更新步幅，再按計畫放大；[1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)已看到方向正確也可能因步子太大而提高代價。這是預先指定的起步方式，暖身長度與高峰仍須用驗證資料選擇。

暖身之後用餘弦曲線平滑降低是一種常見選擇。下面只計時間表；索引i從0開始，i=0就是第一次更新，沒有先跳過一輪。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.training import learning_rate

rates = [learning_rate(i, 40, peak=0.001, warmup=5) for i in range(40)]
print("前六步", rates[:6])
print("最後三步", rates[-3:])
assert max(rates) <= 0.001 + 1e-9

輸入每次傳入步序i、總步數40、高峯與暖身長度。i從0開始，`learning_rate(0,...)` 就是第一次更新的步幅，不是先跳過一輪；這工具在暖身區使用 `(i+1)/warmup` 乘高峯。i=5（第六次更新）時進入下降區開頭，仍在高峯，因此前六項最後兩個都0.001。後面逐漸降低，最後三項接近0.0001，而非硬變零。最後的 `assert` 找出40個值的最大值，核對它沒有超過高峰附近；`1e-9` 是 0.000000001，作為比較時的微小容差，允許浮點計算的捨入差。排程的高峰設定仍是0.001。

圖中橫軸是更新次序，縱軸是當次學習率。排程按預定進度工作，不會自動知道最佳步幅；同樣第20步，總計畫40步或100步就不同。續訓需保存總計畫，延長訓練則是在改計畫。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-05-06-learning-rate.svg")))

練習把warmup從5改成10，先預測首步0.0001、索引4只有0.0005、索引9纔到0.001，再執行核對。最後一段仍向約0.0001降低，但具體路線變了。把前十項印出來，你就能用一排可核對數字確認這份時間表，而不是隻憑排程名稱判斷。

<details>
<summary>補充：實作約定與原始紀錄</summary>

工具保留高峯一成作尾端下限。當總步數至少有4步時，它把暖身長度限制在總步數的四分之一以內；若整趟只有1–3步，暖身長度的上限則是1步。這裡40步最多暖身10步，所以練習從5改10確實生效；若填30並不表示三十步都暖身，實際會被限制成10。讀一個排程要看實現的邊界與索引約定，不能只看到warmup、cosine名字就假定完全相同。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
